In [1]:
import numpy as np
from scipy.stats import chi2
import torch
from sbi.inference import NPE

In [2]:
#in the Prangle paper sigma=exp(2.3), however, 
#here I let the well specified case be sigma=5 so that the effects of misspecification are slightly easier to see
#misspecified settings while (like in the Toy example) involve the simulator assuming sigma=5 when actually it is not
def simulate_lv(x10=50, x20=100, sigma = 5, T=33, max_transitions=100000, seed=2, dat=False,
               return_theta=False):
    #simulation done via the Gillespie method
    #Theta generated from the prior
    rng = np.random.default_rng(seed)
    records = []
    while len(records)<17: #conditioning on non-extinction, as in the Baragatti paper. Could choose to not do this if simulation takes too long
        if dat==True:
            theta1=1
            theta2=0.005
            theta3=0.6
            #the above lead to non extinction runs mostly
        else:
            logtheta1 = rng.uniform(-6, 2)
            logtheta2 = rng.uniform(-6, 2)
            logtheta3 = rng.uniform(-6, 2)
            theta1 = np.exp(logtheta1)
            theta2= np.exp(logtheta2)
            theta3 = np.exp(logtheta3)
        
        t = 0
        x1 = x10
        x2 = x20
    
        checks = list(range(2, T, 2))
        checkpoint=0
        rand1 = rng.normal(loc=0, scale=1)
        rand2 = rng.normal(loc=0, scale=1)
        records = [(
            x1 + sigma*rand1,
            x2 + sigma*rand2
        )]
        transition_count = 0
        while t < T and x1 > 0 and x2 > 0:
            checker = checks[checkpoint]
            
            a1 = theta1 * x1
            a2 = theta2 * x1 * x2
            a3 = theta3 * x2
    
            a0 = a1 + a2 + a3
    
            if a0 == 0:
                break
    
            # waiting time
            tau = rng.exponential(1 / a0) #uses scale, i.e inverse of rate parameter
            t += tau
            while checkpoint < len(checks) and t >= checks[checkpoint]:
                error1 = rng.normal(loc=0, scale=1)
                error2 = rng.normal(loc=0, scale=1)
                rec1 = x1+error1*sigma
                rec2 = x2+error2*sigma
                if rec1 <0:
                    rec1=int(0)
                if rec2<0:
                    rec2=int(0)
                records.append((rec1, rec2))
                checkpoint += 1
    
            if checkpoint == len(checks):
                break
            
            r = rng.random() * a0
            if r < a1:
                x1 += 1
    
            elif r < a1 + a2:
                x1 -= 1
                x2 += 1
    
            else:
                x2 -= 1
        
            transition_count += 1 
            if transition_count > max_transitions:
                return None      # reject this simulation
    if return_theta==True:
        theta=(theta1, theta2, theta3)
        return (theta, np.array(records))
    return np.array(records)

dat = simulate_lv(dat=True, sigma=20)
dat2 = simulate_lv(dat=True)
print(dat2)

[[ 47.76869274 108.44054385]
 [170.3740647   69.8305703 ]
 [322.64117861 336.63323056]
 [ 33.28581139 344.34916875]
 [  8.88492377 147.77707921]
 [ 61.10008511  65.28671638]
 [267.80860203  79.76895538]
 [190.27860996 510.81574946]
 [ 21.21189768 277.02996355]
 [ 29.63419157  69.46551333]
 [ 59.01838414  26.06278357]
 [315.87618914  59.55942633]
 [115.33001082 620.84173166]
 [  0.97193961 233.2928822 ]
 [ 15.44040708  64.71629654]
 [ 14.8056785   19.82805329]
 [125.84287423   3.99230362]]


In [3]:
def simulate_lv_assumed(theta1, theta2, theta3, x10=50, x20=100, sigma=5, T=33,
                         max_transitions=100000, seed=2):
    rng = np.random.default_rng(seed)
    records = []
    while len(records) < 17:
        t = 0
        x1, x2 = x10, x20
        checks = list(range(2, T, 2))
        checkpoint = 0
        rand1, rand2 = rng.normal(size=2)
        records = [(x1 + sigma * rand1, x2 + sigma * rand2)]
        transition_count = 0
        while t < T and x1 > 0 and x2 > 0:
            a1 = theta1 * x1
            a2 = theta2 * x1 * x2
            a3 = theta3 * x2
            a0 = a1 + a2 + a3
            if a0 == 0:
                break
            tau = rng.exponential(1 / a0)
            t += tau
            while checkpoint < len(checks) and t >= checks[checkpoint]:
                e1, e2 = rng.normal(size=2)
                rec1, rec2 = x1 + e1 * sigma, x2 + e2 * sigma
                if rec1 < 0: rec1 = int(0)
                if rec2 < 0: rec2 = int(0)
                records.append((rec1, rec2))
                checkpoint += 1
            if checkpoint == len(checks):
                break
            r = rng.random() * a0
            if r < a1:
                x1 += 1
            elif r < a1 + a2:
                x1 -= 1
                x2 += 1
            else:
                x2 -= 1
            transition_count += 1
            if transition_count > max_transitions:
                return None
    return np.array(records)
dat2= simulate_lv_assumed(theta1=1, theta2=0.005, theta3=0.6)
print(dat2)

[[ 47.76869274 108.44054385]
 [170.3740647   69.8305703 ]
 [322.64117861 336.63323056]
 [ 33.28581139 344.34916875]
 [  8.88492377 147.77707921]
 [ 61.10008511  65.28671638]
 [267.80860203  79.76895538]
 [190.27860996 510.81574946]
 [ 21.21189768 277.02996355]
 [ 29.63419157  69.46551333]
 [ 59.01838414  26.06278357]
 [315.87618914  59.55942633]
 [115.33001082 620.84173166]
 [  0.97193961 233.2928822 ]
 [ 15.44040708  64.71629654]
 [ 14.8056785   19.82805329]
 [125.84287423   3.99230362]]


In [4]:
def simulate_lv_true(theta1, theta2, theta3, K, x10=50, x20=100, sigma=5, T=33,
                      max_transitions=100000, seed=2):
    rng = np.random.default_rng(seed)
    records = []
    while len(records) < 17:  # conditioning on non-extinction, as in simulate_lv
        t = 0
        x1 = x10
        x2 = x20

        checks = list(range(2, T, 2))
        checkpoint = 0
        rand1 = rng.normal(loc=0, scale=1)
        rand2 = rng.normal(loc=0, scale=1)
        records = [(
            x1 + sigma * rand1,
            x2 + sigma * rand2
        )]
        transition_count = 0
        while t < T and x1 > 0 and x2 > 0:
            # Logistic (density-dependent) prey growth hazard -- the misspecified term.
            # Clipped at 0 in case x1 > K (hazard can't go negative).
            a1 = max(theta1 * x1 * (1 - x1 / K), 0.0)
            a2 = theta2 * x1 * x2
            a3 = theta3 * x2

            a0 = a1 + a2 + a3

            if a0 == 0:
                break

            # waiting time
            tau = rng.exponential(1 / a0)  # uses scale, i.e. inverse of rate parameter
            t += tau
            while checkpoint < len(checks) and t >= checks[checkpoint]:
                error1 = rng.normal(loc=0, scale=1)
                error2 = rng.normal(loc=0, scale=1)
                rec1 = x1 + error1 * sigma
                rec2 = x2 + error2 * sigma
                if rec1 < 0:
                    rec1 = int(0)
                if rec2 < 0:
                    rec2 = int(0)
                records.append((rec1, rec2))
                checkpoint += 1

            if checkpoint == len(checks):
                break

            r = rng.random() * a0
            if r < a1:
                x1 += 1
            elif r < a1 + a2:
                x1 -= 1
                x2 += 1
            else:
                x2 -= 1

            transition_count += 1
            if transition_count > max_transitions:
                return None  # reject this simulation

    return np.array(records)

dat = simulate_lv_true(1,0.005,0.6,300, seed=1)
print(dat)

dat2 = simulate_lv_true(1,0.005,0.6,300, seed=10)
print(dat2)

[[ 51.72792096 104.10809072]
 [114.34525891  74.39202171]
 [154.49630714  86.92189308]
 [150.21773518 139.6703594 ]
 [119.35974669 171.03700499]
 [ 87.87298832 144.67831767]
 [111.78781459 106.58185036]
 [ 97.82017678 110.2425621 ]
 [142.90599024  74.45498154]
 [138.89833672  93.70675271]
 [142.42047912 119.96963986]
 [141.67286161 128.46518002]
 [132.49987384 134.30507711]
 [105.71880808 132.97620963]
 [123.95506829 111.34810697]
 [136.20729882 116.99132323]
 [137.15529127 129.6043612 ]]
[[ 44.48330775  96.3748768 ]
 [101.54373255  68.20050464]
 [168.28087629  48.30956533]
 [198.72727352  94.82614183]
 [121.57452069 161.212296  ]
 [ 66.48163622 130.89160612]
 [105.08627943  84.28567279]
 [142.58801913  95.42228495]
 [122.43256332 117.43340676]
 [143.64992286 107.78105883]
 [ 98.18916848 161.86263103]
 [ 98.50192241 108.08045704]
 [136.02752154  79.99440421]
 [167.06443105  96.629971  ]
 [124.02128883 160.88891567]
 [ 92.82781216 158.21933217]
 [ 94.91225631 135.79179772]]


In [5]:
from scipy.optimize import minimize
# ---------------------------------------------------------------------
def full_data_vector(traj):
    return traj.flatten()  # shape (34,)
 
 
def data_distribution(sim_fn, theta, extra_kwargs, n_reps, seed0):
    vecs = []
    for i in range(n_reps):
        traj = sim_fn(*theta, seed=seed0 + i, **extra_kwargs)
        if traj is not None:
            vecs.append(full_data_vector(traj))
    return np.array(vecs)
 
 
# ---------------------------------------------------------------------
# 4. Build the target (true-process) distribution over the 34-dim data
# ---------------------------------------------------------------------
THETA_TRUE = (1.0, 0.005, 0.6)
K_TRUE = 100
N_REPS_TARGET = 600  
 
target_data = data_distribution(
    simulate_lv_true, THETA_TRUE, {"K": K_TRUE}, N_REPS_TARGET, seed0=10_000
)
target_mean = target_data.mean(axis=0)

def objective(log_theta, n_reps=150, seed0=50_000):
    theta = np.exp(log_theta)
    cand_data = data_distribution(simulate_lv_assumed, tuple(theta), {}, n_reps, seed0)
    if len(cand_data) < 0.5 * n_reps:
        return 1e6
    cand_mean = cand_data.mean(axis=0)
    diff = cand_mean - target_mean
    return float(diff @ diff)     
 

x0 = np.log(THETA_TRUE)  # start the search at theta_true itself
res_original = minimize(
    objective, x0, method="Nelder-Mead",
    options={"xatol": 1e-3, "fatol": 1e-2, "maxiter": 300, "disp": True},
)
theta_star = np.exp(res_original.x)
print("\nEstimated pseudo-true theta* (theta1, theta2, theta3):")
print(theta_star)
print(f"\n(True theta was {THETA_TRUE}, K={K_TRUE}, "
      f"which is absent from the assumed model)")


Optimization terminated successfully.
         Current function value: 87371.205380
         Iterations: 52
         Function evaluations: 129

Estimated pseudo-true theta* (theta1, theta2, theta3):
[1.00062874 0.01929948 0.57302982]

(True theta was (1.0, 0.005, 0.6), K=100, which is absent from the assumed model)


In [6]:
THETA_NEW = (0.25,0.005,0.55)
x02 = np.log(THETA_NEW)
res = minimize(
    objective,x02 , method="Nelder-Mead",
    options={"xatol": 1e-3, "fatol": 1e-2, "maxiter": 300, "disp": True},
)
theta_star = np.exp(res.x)
print("\nEstimated pseudo-true theta* (theta1, theta2, theta3):")
print(theta_star)
print(f"\n(True theta was {THETA_TRUE}, K={K_TRUE}, "
      f"which is absent from the assumed model)")

Optimization terminated successfully.
         Current function value: 28694.873533
         Iterations: 52
         Function evaluations: 132

Estimated pseudo-true theta* (theta1, theta2, theta3):
[0.22375171 0.0071137  0.56289875]

(True theta was (1.0, 0.005, 0.6), K=100, which is absent from the assumed model)


In [8]:
print(res.fun)

47680.47601231388


In [7]:
print(res.fun) #round two!

28694.873533118112


In [10]:
print(res_original.fun)

77789.37312382646


In [ ]:
import itertools

starts = [
    (1.0, 0.005, 0.6),      # theta_true
    (0.5, 0.005, 0.6),      # your alt
    #(2.0, 0.005, 0.6),
    (0.3, 0.005, 0.6),
    (0.5, 0.02,0.6)
    #(1.0, 0.005, 1.5),
    #(0.3, 0.001, 0.3),
    #(3.0, 0.01, 1.0),
]

results = []
for s in starts:
    r = minimize(objective, np.log(s), method="Nelder-Mead",
                 options={"xatol": 1e-3, "fatol": 1e-2, "maxiter": 300})
    results.append((s, np.exp(r.x), r.fun))
    print(f"start={s}  ->  theta*={np.exp(r.x)}  obj={r.fun:.2f}")

best_start, theta_star_best, best_obj = min(results, key=lambda t: t[2])
print("\nBest so far:", theta_star_best, "obj =", best_obj)

start=(1.0, 0.005, 0.6)  ->  theta*=[1.00062874 0.01929948 0.57302982]  obj=87371.21
start=(0.5, 0.005, 0.6)  ->  theta*=[0.4769514  0.00884505 0.58591371]  obj=45920.31


In [3]:
def gen_ref_table(N=10000): #upped samples when run on the cluster
    #takes around 30 seconds for 1000 entries
    ref_table = list()
    for i in range(N):
        pair = simulate_lv(sigma=5, return_theta=True, seed=i+1) #reminder: misspecification will not come from the prior here
        if pair: #sometimes max_iterations are exceeded. In this case we will have fewer than N pairs
            ref_table.append(pair) #uses a 34 dimensional summary. May suffer from curse of dimensionality here
    return ref_table

def reject_abc(ref_table, data, quantile=0.01):
    """
    ref_table: iterable of (theta, sims), where:
               theta has shape (3,)
               sims has shape (17, 2)
    data: array-like of shape (17, 2)
    quantile: quantile used to determine epsilon
    """
    data = np.asarray(data)
    sims = np.asarray([entry[1] for entry in ref_table])
    # Squared Euclidean distance for each simulation
    distances = np.sum((sims - data) ** 2, axis=(1, 2))
    # ABC threshold
    epsilon = np.quantile(distances, quantile)
    # Keep simulations below threshold
    accepted = distances <= epsilon
    return [
        (ref_table[i][0], ref_table[i][1])
        for i in np.flatnonzero(accepted)
    ]

In [18]:
def abc_confregion(ref_table, data, alpha=0.1): #again, a 90% credible region is the goal
    accepted = reject_abc(ref_table, data)
    #Now using these accepted samples to define a credible region
    theta_candidates = np.array([theta for theta, simulation in accepted])
    #Will use the multivariate normal ellipsoid method as in Baragatti et. al
    # Posterior mean
    mu = theta_candidates.mean(axis=0)
    # Posterior covariance
    Sigma = np.cov(theta_candidates, rowvar=False)
    threshold = chi2.ppf(1-alpha, df=3)
    return (mu, Sigma, threshold) #defines a (100*(1-alpha))% credible region under a Gaussian approximation of the posterior

def robnpe(ref_table, data, alpha=0.1, abc_quantile=0.01):
    ref_table2 = reject_abc(ref_table, data, quantile=abc_quantile)
    theta_train = np.array([theta for theta, sim in ref_table2])
    x_train = np.array([sim for theta, sim in ref_table2])
    # Flatten (17, 2) -> 34-dimensional vector
    x_train = x_train.reshape(len(x_train), -1)
    theta_train = torch.tensor(theta_train, dtype=torch.float32)
    x_train = torch.tensor(x_train, dtype=torch.float32)

    inference = NPE()  # uses default neural spline flow approach
    _ = inference.append_simulations(theta_train, x_train).train()
    posterior = inference.build_posterior()

    data_flat = np.asarray(data).reshape(-1)
    data_flat = torch.as_tensor(data_flat, dtype=torch.float32)
    posterior.set_default_x(data_flat)

    posterior_samples = posterior.sample((1000,)).numpy()  # shape (1000, 3)
    mu = posterior_samples.mean(axis=0)
    Sigma = np.cov(posterior_samples, rowvar=False)
    threshold = chi2.ppf(1-alpha, df=3)
    return (mu, Sigma, threshold) #defines a (100*(1-alpha))% credible region under a Gaussian approximation of the posterior

def npe(posterior, data, alpha=0.1):
    data_flat = np.asarray(data).reshape(-1)
    data_flat = torch.as_tensor(data_flat, dtype=torch.float32)
    posterior.set_default_x(data_flat)

    posterior_samples = posterior.sample((1000,)).numpy()  # shape (1000, 3)
    mu = posterior_samples.mean(axis=0)
    Sigma = np.cov(posterior_samples, rowvar=False)
    threshold = chi2.ppf(1-alpha, df=3)
    return (mu, Sigma, threshold) #defines a (100*(1-alpha))% credible region under a Gaussian approximation of the posterior

def is_in_ellipsoid(theta, mu, Sigma, threshold):
    theta = np.asarray(theta)
    diff = theta - mu
    Sigma_inv = np.linalg.inv(Sigma)
    mahalanobis_sq = diff @ Sigma_inv @ diff
    return mahalanobis_sq <= threshold
    
def ellipsoid_volume_3d(mu, Sigma, threshold):
    vol = (4/3) * np.pi * (threshold ** 1.5) * np.sqrt(np.linalg.det(Sigma))
    return vol


In [21]:
def robconf(ref_table, data, train_prop=0.8, alpha=0.1, abc_quantile1=0.0025, abc_quantile2=0.01):
    train_size = round(train_prop * len(ref_table))
    train_table = ref_table[:train_size]
    calib_vanilla = ref_table[train_size:]
    ref_table2 = reject_abc(train_table, data, quantile=abc_quantile1)
    theta_train = np.array([theta for theta, sim in ref_table2])
    x_train = np.array([sim for theta, sim in ref_table2])
    
    # Flatten (17, 2) -> 34-dimensional vector
    x_train = x_train.reshape(len(x_train), -1)
    theta_train = torch.tensor(theta_train, dtype=torch.float32)
    x_train = torch.tensor( x_train,dtype=torch.float32)

    inference = NPE()
    _ = inference.append_simulations(
        theta_train,
        x_train
    ).train()
    posterior = inference.build_posterior()

    material = reject_abc(
        calib_vanilla,
        data,
        quantile=abc_quantile2
    )
    theta_array = np.array([
        theta for theta, sim in material
    ])
    summary_abc = np.array([
        sim for theta, sim in material
    ])
    # Flatten (17, 2) -> 34-dimensional vector
    summary_abc = summary_abc.reshape(
        len(summary_abc), -1
    )
    theta_abc = torch.tensor(
        theta_array,
        dtype=torch.float32
    )
    summaries_torch = torch.tensor(
        summary_abc,
        dtype=torch.float32
    )
    # -------------------------------------------------
    # Calculate robust conformal cutoff
    # -------------------------------------------------
    scores = -posterior.log_prob_batched(
        theta_abc,
        summaries_torch
    ).detach().numpy()

    scores = np.ravel(scores)
    q_level = 1.0 - alpha
    robglob_cutoff = np.quantile( #could make this dependent on sample size
        scores,
        q_level
    )
    # -------------------------------------------------
    # Score the ABC-selected training simulations
    # -------------------------------------------------

    scores = -posterior.log_prob_batched(
        theta_train,
        x_train
    ).detach().numpy()

    scores = np.ravel(scores)

    # -------------------------------------------------
    # Construct conformal region
    # -------------------------------------------------

    accepted_thetas = theta_train.numpy()[
        scores <= robglob_cutoff
    ]

    if len(accepted_thetas) == 0:
        return None

    return (posterior, robglob_cutoff, theta_train)

def isin_robconfregion(theta, posterior, data, cutoff, theta_train_np):
    theta = np.asarray(theta).reshape(-1)
    t_min = np.asarray(theta_train_np).min(axis=0).reshape(-1)
    t_max = np.asarray(theta_train_np).max(axis=0).reshape(-1)

    if not all(float(t_min[i]) <= float(theta[i]) <= float(t_max[i]) for i in range(3)):
        return False

    theta_t = torch.tensor(theta, dtype=torch.float32)
    data_flat = np.asarray(data).reshape(-1)          # (17,2) -> (34,)
    data_t = torch.tensor(data_flat, dtype=torch.float32)

    score = -posterior.log_prob(theta_t, data_t).detach().item()
    return score <= cutoff

def robconfregion_volume(posterior, data, cutoff, theta_train_np, n_samples=200_000, seed=None):
    theta_train_np = np.asarray(theta_train_np)   # handles torch tensor or numpy array alike
    rng = np.random.default_rng(seed)
    t_min = theta_train_np.min(axis=0)
    t_max = theta_train_np.max(axis=0)
    box_volume = np.prod(t_max - t_min)
    return box_volume #note that this is technically an overestimate of the conformal region's volume

    '''
    samples = rng.uniform(t_min, t_max, size=(n_samples, len(t_min)))
    samples_t = torch.tensor(samples, dtype=torch.float32)

    data_flat = np.asarray(data).reshape(-1)
    data_t = torch.tensor(data_flat, dtype=torch.float32)
    data_batch = data_t.unsqueeze(0).expand(n_samples, -1)

    with torch.no_grad():
        scores = -posterior.log_prob_batched(samples_t, data_batch).numpy()
        scores = np.ravel(scores)
    frac_inside = np.mean(scores <= cutoff)
    volume_estimate = frac_inside * box_volume
    se = box_volume * np.sqrt(frac_inside * (1 - frac_inside) / n_samples)
    return volume_estimate
    '''


In [38]:
def fit_npe(ref_table): #once trained can be used again and again on new data (amortised)
    theta_train = np.array([theta for theta, sim in ref_table])
    x_train = np.array(
        [x.reshape(-1) for theta, x in ref_table]
    )
    theta_train = torch.tensor(theta_train, dtype=torch.float32)
    x_train = torch.tensor(x_train, dtype=torch.float32)

    inference = NPE() #uses default neural spline flow approach
    _ = inference.append_simulations(
        theta_train,
        x_train
    ).train()
    posterior = inference.build_posterior()
    return posterior

def fit_global_conformal_cutoff(posterior, calib_table, alpha=0.1):
    scores = []
    for theta, sim in calib_table:
        theta_torch = torch.tensor(
            theta,
            dtype=torch.float32
        ).unsqueeze(0)
        sim_torch = torch.tensor(
            sim.reshape(-1),
            dtype=torch.float32
        ).unsqueeze(0)
        score = -posterior.log_prob(
            theta_torch,
            x=sim_torch
        ).item()
        scores.append(score)

    B = len(scores)
    q_level = min(
        (1 + 1/B)*(1-alpha),
        1.0
    )
    return np.quantile(scores, q_level)

def in_conformalregion(theta, posterior, cutoff, data):
    theta = np.asarray(theta).reshape(-1)
    theta_t = torch.tensor(theta, dtype=torch.float32)
    data_flat = np.asarray(data).reshape(-1)  # (17,2) -> (34,)
    data_t = torch.tensor(data_flat, dtype=torch.float32)
    score = -posterior.log_prob(theta_t, data_t).detach().item()
    return score <= cutoff

def sample_prior(n, seed=None):
    rng = np.random.default_rng(seed)
    log_theta = rng.uniform(-6, 2, size=(n, 3))
    return np.exp(log_theta)

proposed_theta = sample_prior(100000, seed=123) 

def conformalregion_volume(posterior, cutoff, data, theta_candidates=proposed_theta):
    x_obs = torch.tensor(
        data.reshape(-1),
        dtype=torch.float32
    )
    theta_torch = torch.tensor(theta_candidates, dtype=torch.float32)
    scores = -posterior.log_prob(
        theta_torch,
        x=x_obs
    ).detach().numpy()
    accepted_theta = theta_candidates[scores <= cutoff]
    t_min = accepted_theta.min(axis=0)
    t_max = accepted_theta.max(axis=0)
    box_volume = np.prod(t_max - t_min)
    return box_volume #this approach would break down completely with multimodal posteriors


In [52]:
ref_table = gen_ref_table(25000)
train_size = round(0.8*len(ref_table))
train_table = ref_table[:train_size] #used to train the Conformal NPE
calib_table = ref_table[train_size:] 

theta_train = np.array([theta for theta, sim in ref_table])
x_train = np.array([sim for theta, sim in ref_table])
x_train = x_train.reshape(len(x_train), -1)
theta_train = torch.tensor(theta_train, dtype=torch.float32)
x_train = torch.tensor(x_train, dtype=torch.float32)
inference = NPE()  # uses default neural spline flow approach
_ = inference.append_simulations(theta_train, x_train).train()
posterior_npe = inference.build_posterior()

posterior_conformal = fit_npe(train_table)
global_cutoff = fit_global_conformal_cutoff(posterior=posterior_conformal, 
                                            calib_table=calib_table,
                                            alpha=0.1)

def assess_coverage(dat_iters = 1000, alpha=0.1, sd=5):
    totcov_abc = 0
    totvolume_abc = 0
    totcov_robconf = 0
    totvolume_robconf = 0
    totcov_robnpe = 0
    totvolume_robnpe = 0 
    totcov_npe = 0
    totvolume_npe = 0
    totcov_globconf=0
    totvolume_globconf = 0
    total_experiments = 0
    for k in range(dat_iters):
        lv_output = simulate_lv(seed=k+1,sigma=sd,return_theta=True)
        if lv_output:
            theta_true, obs_data = lv_output
            total_experiments+=1
        else:
            continue
            
        #Coverage Check ABC and volume of ellipsoid
        abc_region_mu, abc_region_Sigma, abc_region_threshold = abc_confregion(
            ref_table=ref_table, data=obs_data, alpha=alpha)
        if is_in_ellipsoid(theta_true, abc_region_mu, abc_region_Sigma, abc_region_threshold):
            totcov_abc+=1
        totvolume_abc += ellipsoid_volume_3d(abc_region_mu, abc_region_Sigma, abc_region_threshold)

        #Coverage Check NPE and volume of ellipsoid
        npe_region_mu, npe_region_Sigma, npe_region_threshold = npe(posterior=posterior_npe,
                                                                    data=obs_data)
        if is_in_ellipsoid(theta_true, npe_region_mu, npe_region_Sigma, npe_region_threshold):
            totcov_npe+=1
        totvolume_npe += ellipsoid_volume_3d(npe_region_mu, npe_region_Sigma, npe_region_threshold)

        #Coverage Check RobNPE and volume of ellipsoid
        robnpe_region_mu, robnpe_region_Sigma, robnpe_region_threshold = robnpe(
            ref_table=ref_table, data=obs_data)
        if is_in_ellipsoid(theta_true, robnpe_region_mu, robnpe_region_Sigma, robnpe_region_threshold):
            totcov_robnpe+=1
        totvolume_robnpe += ellipsoid_volume_3d(robnpe_region_mu, robnpe_region_Sigma, robnpe_region_threshold)

        #Coverage Check GlobConf and approx. volume (overestimate) of conformal region
        if in_conformalregion(theta_true, posterior_conformal, global_cutoff, obs_data):
            totcov_globconf+=1
        totvolume_globconf+= conformalregion_volume(posterior=posterior_conformal, cutoff=global_cutoff, data=obs_data)

        #Coverage Check RobConf and approx. volume (overestimate) of conformal region
        robposterior, robcutoff, robthetas = robconf(ref_table=ref_table, data=obs_data) 
        if isin_robconfregion(theta_true, robposterior, data=obs_data, cutoff=robcutoff, theta_train_np=robthetas):
            totcov_robconf+=1
        totvolume_robconf += robconfregion_volume(posterior=robposterior, data=obs_data, 
            cutoff=robcutoff, theta_train_np=robthetas, n_samples=100, seed=17) #slightly random but hopefully not too bad
        
    cov_abc = totcov_abc/total_experiments
    meanvol_abc = totvolume_abc/total_experiments
    cov_npe = totcov_npe/total_experiments
    meanvol_npe = totvolume_npe/total_experiments
    cov_robnpe = totcov_robnpe/total_experiments
    meanvol_robnpe = totvolume_robnpe/total_experiments
    cov_robconf = totcov_robconf/total_experiments
    meanvol_robconf = totvolume_robconf/total_experiments
    cov_globconf = totcov_globconf/total_experiments
    meanvol_globconf = totvolume_globconf/total_experiments
    
    return (cov_abc, cov_robconf, cov_robnpe, cov_npe, cov_globconf,
            meanvol_abc, meanvol_robconf, meanvol_robnpe, meanvol_npe, meanvol_globconf,
           total_experiments)


overspec = assess_coverage(dat_iters=500, sd=1)
wellspec = assess_coverage(dat_iters=500)
underspec1 = assess_coverage(dat_iters=500, sd=10)
underspec2 = assess_coverage(dat_iters=500, sd=20)


  0%|          | 0/1000 [00:00<?, ?it/s]

 Neural network successfully converged after 89 epochs.

  0%|          | 0/1000 [00:00<?, ?it/s]

 Neural network successfully converged after 89 epochs.

  0%|          | 0/1000 [00:00<?, ?it/s]

...

 Neural network successfully converged after 95 epochs.

  0%|          | 0/1000 [00:00<?, ?it/s]

 Neural network successfully converged after 134 epochs.

  0%|          | 0/1000 [00:00<?, ?it/s]

RuntimeError: can't start new thread

In [53]:
wellspec

(0.9361702127659575,
 0.9042553191489362,
 0.9893617021276596,
 0.9361702127659575,
 0.9574468085106383,
 np.float64(0.010500504584620037),
 np.float32(0.043291572),
 np.float64(0.3455097006574665),
 np.float64(0.0795128824465568),
 np.float64(0.02647862165437519))

In [54]:
overspec

(0.925531914893617,
 0.9042553191489362,
 0.9468085106382979,
 0.9361702127659575,
 0.9468085106382979,
 np.float64(0.010783502367224284),
 np.float32(0.044943836),
 np.float64(0.040716199005052474),
 np.float64(0.2815671636304668),
 np.float64(0.026306718301573852))

In [55]:
underspec

(0.9361702127659575,
 0.8936170212765957,
 0.9787234042553191,
 0.9468085106382979,
 0.9361702127659575,
 np.float64(0.01028168988850253),
 np.float32(0.043326866),
 np.float64(0.5771437742446958),
 np.float64(0.06903528734698866),
 np.float64(0.02604595344428161))

In [56]:
underspec2

(0.925531914893617,
 0.8829787234042553,
 0.9042553191489362,
 0.925531914893617,
 0.8404255319148937,
 np.float64(0.010603854873269208),
 np.float32(0.045428447),
 np.float64(0.6993134155895449),
 np.float64(0.024254614154291677),
 np.float64(0.0271668696964902),
 94)